In [1]:
#!/usr/bin/env python3
"""
Extract matched audio files from cover and stego RAR archives based on Excel metadata.
Ensures equal distribution across embedding rates and languages.
"""

import pandas as pd
import os
import subprocess
from pathlib import Path
from collections import defaultdict
import shutil

# Configuration
EXCEL_FILE = "I:\\FINAL DATASET\\train_lable.xlsx"  # Update with your Excel file path
COVER_RAR = "I:\\FINAL DATASET\\g729a_0.rar"  # Update with path to cover RAR
STEGO_RAR = "I:\\FINAL DATASET\\g729a_Steg.rar"  # Update with path to stego RAR
OUTPUT_DIR = "I:\\FINAL DATASET\\extracted_files"
TOTAL_FILES = 10000

# Tabs to process
TABS = [
    "train_lable_english_CNV",
    "train_lable_english_PMS", 
    "train_lable_chinese_CNV",
    "train_lable_chinese_PMS"
]

# Embedding rates
EMBEDDING_RATES = [0.1, 0.2, 0.3, 0.4]


def extract_rar(rar_file, output_dir):
    """Extract RAR file to output directory."""
    print(f"Extracting {rar_file}...")
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Try WinRAR command line (common on Windows)
        subprocess.run(['UnRAR.exe', 'x', '-y', rar_file, output_dir], 
                      check=True, capture_output=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        try:
            # Try 7-Zip command line
            subprocess.run(['7z', 'x', f'-o{output_dir}', '-y', rar_file], 
                          check=True, capture_output=True)
        except (subprocess.CalledProcessError, FileNotFoundError):
            try:
                # Try unrar (if installed)
                subprocess.run(['unrar', 'x', '-y', rar_file, output_dir], 
                              check=True, capture_output=True)
            except (subprocess.CalledProcessError, FileNotFoundError):
                print(f"Error: Could not extract {rar_file}.")
                print("Please install one of: WinRAR, 7-Zip, or unrar")
                raise
    
    print(f"Extracted {rar_file} successfully")


def load_excel_data(excel_file):
    """Load all tabs from Excel file."""
    print(f"Loading {excel_file}...")
    data = {}
    
    for tab in TABS:
        df = pd.read_excel(excel_file, sheet_name=tab)
        data[tab] = df
        print(f"  {tab}: {len(df)} rows")
    
    return data


def organize_files_by_category(data):
    """Organize file names by language, algorithm, and embedding rate."""
    organized = defaultdict(lambda: defaultdict(list))
    
    for tab_name, df in data.items():
        # Parse tab name to get language and algorithm
        parts = tab_name.replace("train_lable_", "").split("_")
        language = parts[0]  # english or chinese
        algorithm = parts[1]  # CNV or PMS
        
        # Group by embedding rate
        for _, row in df.iterrows():
            if pd.notna(row.iloc[0]) and pd.notna(row.iloc[2]):  # Check if file and rate exist
                filename = str(row.iloc[0]).strip() + ".g729a"
                rate = float(row.iloc[2])
                
                key = (language, algorithm, rate)
                organized[key]['files'].append(filename)
    
    return organized


def calculate_distribution(total_files, categories):
    """Calculate how many files to select from each category."""
    # Total categories: 2 languages × 2 algorithms × 4 rates = 16 categories
    num_categories = len(categories)
    files_per_category = total_files // num_categories
    remainder = total_files % num_categories
    
    distribution = {}
    for i, category in enumerate(sorted(categories.keys())):
        # Distribute remainder across first few categories
        count = files_per_category + (1 if i < remainder else 0)
        distribution[category] = count
    
    return distribution


def select_files(organized, distribution):
    """Select files according to distribution."""
    import random   # add this if not already imported
    selected = defaultdict(list)
    
    for category, count in distribution.items():
        available = organized[category]['files']

        # shuffle to avoid bias
        random.shuffle(available)
        
        if len(available) < count:
            print(f"Warning: {category} has only {len(available)} files, need {count}")
            selected[category] = available
        else:
            # Take first 'count' files after shuffle
            selected[category] = available[:count]
    
    return selected


def copy_matching_files(selected, cover_dir, stego_dir, output_dir):
    """Copy matching files from cover and stego directories."""
    cover_output = os.path.join(output_dir, "cover")
    stego_output = os.path.join(output_dir, "stego")
    
    os.makedirs(cover_output, exist_ok=True)
    os.makedirs(stego_output, exist_ok=True)
    
    total_copied = 0
    missing_files = []
    
    for category, files in selected.items():
        language, algorithm, rate = category
        print(f"\nProcessing {language}_{algorithm}_rate{rate}: {len(files)} files")
        
        for filename in files:
            # Find files in extracted directories
            cover_file = find_file(cover_dir, filename)
            stego_file = find_file(stego_dir, filename)
            
            if cover_file and stego_file:

                print(f"Copying {filename} from cover: {cover_file} and stego: {stego_file}")

                # Copy with original filename
                shutil.copy2(cover_file, os.path.join(cover_output, filename))
                shutil.copy2(stego_file, os.path.join(stego_output, filename))
                total_copied += 1
            else:
                missing_files.append((filename, bool(cover_file), bool(stego_file)))
    
    print(f"\n{'='*60}")
    print(f"Total files copied: {total_copied} pairs")
    print(f"Cover files: {total_copied}")
    print(f"Stego files: {total_copied}")
    
    if missing_files:
        print(f"\nWarning: {len(missing_files)} files not found:")
        for fname, has_cover, has_stego in missing_files[:10]:
            status = []
            if not has_cover: status.append("cover missing")
            if not has_stego: status.append("stego missing")
            print(f"  {fname}: {', '.join(status)}")
        if len(missing_files) > 10:
            print(f"  ... and {len(missing_files) - 10} more")
    
    return total_copied


def find_file(root_dir, filename):
    """Recursively find a file in directory tree."""
    for dirpath, _, filenames in os.walk(root_dir):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None


def main():
    print(f"{'='*60}")
    print("Audio File Extraction Script")
    print(f"{'='*60}\n")
    
    # Paths to manually extracted folders
    temp_cover = "I:\\FINAL DATASET\\g729a_0"  # manually extracted cover
    temp_stego = "I:\\FINAL DATASET\\g729a_Steg"  # manually extracted stego
    
    try:
        # # Extract RAR files
        # extract_rar(COVER_RAR, temp_cover)
        # extract_rar(STEGO_RAR, temp_stego)
        
        # Load Excel data
        data = load_excel_data(EXCEL_FILE)
        
        # Organize files by category
        print("\nOrganizing files by category...")
        organized = organize_files_by_category(data)
        
        # Show available files per category
        print(f"\n{'='*60}")
        print("Available files per category:")
        print(f"{'='*60}")
        for category in sorted(organized.keys()):
            language, algorithm, rate = category
            count = len(organized[category]['files'])
            print(f"{language:8} {algorithm:3} {rate:3.1f}: {count:6} files")
        
        # Calculate distribution
        print(f"\n{'='*60}")
        print(f"Calculating distribution for {TOTAL_FILES} total files...")
        distribution = calculate_distribution(TOTAL_FILES, organized)
        
        print(f"\n{'='*60}")
        print("Target distribution:")
        print(f"{'='*60}")
        for category in sorted(distribution.keys()):
            language, algorithm, rate = category
            count = distribution[category]
            print(f"{language:8} {algorithm:3} {rate:3.1f}: {count:6} files")
        
        # Select files
        print(f"\n{'='*60}")
        print("Selecting files...")
        selected = select_files(organized, distribution)
        
        # Copy matching files
        print(f"\n{'='*60}")
        print("Copying files to output directory...")
        total_copied = copy_matching_files(selected, temp_cover, temp_stego, OUTPUT_DIR)
        
        print(f"\n{'='*60}")
        print("COMPLETED SUCCESSFULLY!")
        print(f"{'='*60}")
        print(f"Output directory: {OUTPUT_DIR}/")
        print(f"  - cover/: {total_copied} files")
        print(f"  - stego/: {total_copied} files")
        
    finally:
        # # Cleanup temp directories
        # print(f"\nCleaning up temporary directories...")
        # if os.path.exists(temp_cover):
        #     shutil.rmtree(temp_cover)
        # if os.path.exists(temp_stego):
        #     shutil.rmtree(temp_stego)
        print("Cleanup complete")


if __name__ == "__main__":
    main()

Audio File Extraction Script

Loading I:\FINAL DATASET\train_lable.xlsx...
  train_lable_english_CNV: 125924 rows
  train_lable_english_PMS: 126044 rows
  train_lable_chinese_CNV: 65223 rows
  train_lable_chinese_PMS: 65105 rows

Organizing files by category...

Available files per category:
chinese  CNV 0.1:  15670 files
chinese  CNV 0.2:  16915 files
chinese  CNV 0.3:  15576 files
chinese  CNV 0.4:  17062 files
chinese  PMS 0.1:  15526 files
chinese  PMS 0.2:  17013 files
chinese  PMS 0.3:  15637 files
chinese  PMS 0.4:  16929 files
english  CNV 0.1:  30205 files
english  CNV 0.2:  32783 files
english  CNV 0.3:  30298 files
english  CNV 0.4:  32638 files
english  PMS 0.1:  30351 files
english  PMS 0.2:  32683 files
english  PMS 0.3:  30239 files
english  PMS 0.4:  32771 files

Calculating distribution for 10000 total files...

Target distribution:
chinese  CNV 0.1:    625 files
chinese  CNV 0.2:    625 files
chinese  CNV 0.3:    625 files
chinese  CNV 0.4:    625 files
chinese  PMS 0